# 06 - Enrichment

Verification of ICCU-ISTAT integration and of the resulting municipal analytical dataset.

### Reproducibility

This notebook documents and verifies the integration and enrichment phase of ICCU data with ISTAT demographic indicators.

The transformations use exclusively the RAW data and semantic inputs stored in the repository and are implemented in the scripts in the `scripts/` directory.

The notebook allows joins, coverage, granularity, and quality of the final analytical dataset to be verified.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## ICCU + ISTAT enrichment and analytical dataset
The territorial key is `istat_code`; the ICCU → ISTAT 2025 join is deterministic and measured.

In [2]:
lib = pd.read_csv(ROOT / "data/processed/library.csv", dtype=str, keep_default_na=False, low_memory=False)
status = pd.read_csv(ROOT / "data/processed/library_status.csv", dtype=str, keep_default_na=False)
pop = pd.read_csv(ROOT / "data/processed/municipality_population.csv", dtype=str, keep_default_na=False)
ana = pd.read_csv(ROOT / "data/processed/analysis_municipality.csv", dtype=str, keep_default_na=False)
coverage = lib["istat_code"].isin(set(pop["istat_code"])).mean()
assert coverage == 1.0
print(f"Join ICCU → ISTAT 2025: {coverage:.0%} ({len(lib):,}/{len(lib):,})")

Join ICCU → ISTAT 2025: 100% (19,611/19,611)


In [3]:
for c in ["total_registry_records","total_libraries","main_problematic_libraries"]:
    ana[c] = pd.to_numeric(ana[c], errors="raise")
summary = {
    "registry_records": int(ana["total_registry_records"].sum()),
    "analytical_libraries": int(ana["total_libraries"].sum()),
    "main_problematic_libraries": int(ana["main_problematic_libraries"].sum()),
}
assert summary == {"registry_records":19611,"analytical_libraries":18956,"main_problematic_libraries":2497}
summary

{'registry_records': 19611,
 'analytical_libraries': 18956,
 'main_problematic_libraries': 2497}

In [4]:
joinq = pd.read_csv(ROOT / "reports/join_quality.csv", dtype=str, keep_default_na=False)
joinq

,join,master_records,matched_master_records,unmatched_master_records,duplicate_relation_rows,coverage_percent,key,notes
0,library→library_type,19611,13715,5896,0,69.93524042629137,isil,1:1 sulla copertura presente
1,library→library_holdings,19611,13715,5896,79797,69.93524042629137,isil,1:N preservata
2,library→special_collection,19611,2449,17162,7288,12.487889449798582,isil,1:N preservata
3,library→library_contact,19611,13630,5981,49174,69.50181020855642,isil,1:N preservata
4,library→library_previous_name,19611,6584,13027,2923,33.5729947478456,isil,1:N preservata
5,ICCU library→ISTAT 2025 comune,19611,19611,0,0,100.0,istat_code,join su codice ufficiale; no fuzzy matching
6,ISTAT 2025 comuni→crosswalk 2019/2025,7896,7896,0,59,100.0,current_istat_code,righe multiple per predecessori in fusioni/inc...


## Reproduction of datasets from RAW data

The tabular pipeline can be regenerated directly from the original archives available in the repository.

From the project root:

```bash
python scripts/build_processed_data.py \
  --iccu data/raw/iccu/opendata.zip \
  --posas2019 data/raw/istat/POSAS_2019_it_Tutti_i_file.zip \
  --posas2025 data/raw/istat/POSAS_2025_it_Tutti_i_file.zip \
  --out-root .

python scripts/build_metadata.py \
  --root . \
  --iccu data/raw/iccu/opendata.zip \
  --posas2019 data/raw/istat/POSAS_2019_it_Tutti_i_file.zip \
  --posas2025 data/raw/istat/POSAS_2025_it_Tutti_i_file.zip \
  --cultural-on data/external/cultural-ON.owl

python scripts/validate_outputs.py --root .